In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def preprocess_data(train_df, test_df):
    """
    銀行顧客離脱データセットの前処理および特徴量エンジニアリングを行う関数
    """
    print("前処理を開始します...")
    
    # 1. 正解ラベルの分離とIDの退避
    y_train = train_df['Exited'] if 'Exited' in train_df.columns else None
    
    # 学習用とテスト用を区別するためのフラグを立てて一時的に結合
    train_features = train_df.drop(columns=['Exited']) if 'Exited' in train_df.columns else train_df.copy()
    test_features = test_df.copy()
    
    train_features['is_train'] = 1
    test_features['is_train'] = 0
    
    combined = pd.concat([train_features, test_features], axis=0).reset_index(drop=True)
    
    # 2. 名字（Surname）の数（Frequency Encoding）
    # 同じ苗字がデータ中に何回登場するか（家族ルールの抽出）
    surname_counts = combined['Surname'].value_counts()
    combined['Surname_Freq'] = combined['Surname'].map(surname_counts)
    
    # 3. 製品数（NumOfProducts）の非線形性を捉えるフラグ
    # クロス集計で解約率が極端だった部分を明示的にフラグ化
    combined['Is_Product_2'] = (combined['NumOfProducts'] == 2).astype(int)
    combined['Is_High_Product'] = (combined['NumOfProducts'] >= 3).astype(int)
    
    # 4. 残高（Balance）に関する特徴量
    # 残高が0かどうかのフラグ
    combined['Is_Balance_Zero'] = (combined['Balance'] == 0).astype(int)
    # 年収に対する残高の割合（分母の0割りを防ぐため +1）
    combined['Balance_to_Salary_Ratio'] = combined['Balance'] / (combined['EstimatedSalary'] + 1)
    
    # 5. 年齢と信用スコアの掛け合わせ
    combined['CreditScore_by_Age'] = combined['CreditScore'] / combined['Age']
    
    # 6. 交互作用特徴量（国 × 製品数）
    # 「ドイツの製品数3」のような強力な組み合わせを文字列として結合
    combined['Geo_Product'] = combined['Geography'].astype(str) + "_" + combined['NumOfProducts'].astype(str)
    
    # 7. カテゴリ変数のエンコーディング（Label Encoding）
    # GBDTで扱えるように文字列を数値（0, 1, 2...）に変換
    cat_cols = ['Geography', 'Gender', 'Geo_Product']
    for col in cat_cols:
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col].astype(str))
    
    # 8. モデルの学習に不要なカラムの削除
    # 変換元のSurname、一意な値であるCustomerId、行識別用のidは削除（idは後で提出に使うため別途保持）
    drop_cols = ['id', 'CustomerId', 'Surname']
    combined = combined.drop(columns=[col for col in drop_cols if col in combined.columns])
    
    # 9. 再び train と test に分離
    X_train = combined[combined['is_train'] == 1].drop(columns=['is_train']).reset_index(drop=True)
    X_test = combined[combined['is_train'] == 0].drop(columns=['is_train']).reset_index(drop=True)
    
    print(f"前処理が完了しました。特徴量数: {X_train.shape[1]}")
    return X_train, y_train, X_test


In [9]:
# --- ここが原因！関数を実際に「実行」して変数を受け取るコードが必要です ---

# 1. まず元のデータを読み込む（パスはご自身の環境に合わせてください）
train = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\train.csv")
test = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\test.csv")
test_ids = test['id']

# 2. 定義した関数をここで呼び出す（これで X_train たちが誕生します）
X_train, y_train, X_test = preprocess_data(train, test)

前処理を開始します...
前処理が完了しました。特徴量数: 17


In [4]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# 必要モデルのインポート
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

# --- 1. データの読み込み ---
train = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\train.csv")
test = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\test.csv")
test_ids = test['id']

# --- 2. 前処理の実行 ---
X_train, y_train, X_test = preprocess_data(train, test)

# --- 3. クロスバリデーションの設定（Stratified K-Fold） ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 各モデルのOOF予測値とテスト予測値の格納用
models_info = {
    'LightGBM':     {'oof': np.zeros(len(X_train)), 'test': np.zeros(len(X_test))},
    'XGBoost':      {'oof': np.zeros(len(X_train)), 'test': np.zeros(len(X_test))},
    'CatBoost':     {'oof': np.zeros(len(X_train)), 'test': np.zeros(len(X_test))},
    'RandomForest': {'oof': np.zeros(len(X_train)), 'test': np.zeros(len(X_test))}
}

# カテゴリ変数のカラム名
cat_cols = ['Geography', 'Gender', 'Geo_Product']

# 保存先の設定
save_dir = "../../model/"
os.makedirs(save_dir, exist_ok=True)
prefix = "05251045_"

print("=== 4大モデルの交差検証（5-Fold）を開始します ===")

# --- 4. 学習と予測のループ ---
for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n▼▼▼ Fold {fold + 1} / 5 の学習開始 ▼▼▼")
    
    # データの分割
    X_tr = X_train.iloc[train_idx].copy()
    y_tr = y_train.iloc[train_idx]
    X_va = X_train.iloc[valid_idx].copy()
    y_va = y_train.iloc[valid_idx]
    X_te = X_test.copy()
    
    # ----------------------------------------------------
    # 1/4: LightGBM
    # ----------------------------------------------------
    print("1/4: LightGBM...")
    for col in cat_cols:
        X_tr[col] = X_tr[col].astype('category')
        X_va[col] = X_va[col].astype('category')
        X_te[col] = X_te[col].astype('category')
        
    trn_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_va, label=y_va)
    
    model_lgb = lgb.train(
        {'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt', 'learning_rate': 0.05, 'random_state': 42, 'verbose': -1},
        trn_data, num_boost_round=1000, valid_sets=[trn_data, val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    models_info['LightGBM']['oof'][valid_idx] = model_lgb.predict(X_va, num_iteration=model_lgb.best_iteration)
    models_info['LightGBM']['test'] += model_lgb.predict(X_te, num_iteration=model_lgb.best_iteration) / cv.n_splits
    joblib.dump(model_lgb, f"{save_dir}{prefix}lightgbm_fold{fold+1}.pkl")

    # ----------------------------------------------------
    # 2/4: XGBoost
    # ----------------------------------------------------
    print("2/4: XGBoost...")
    # 【修正】DMatrixの引数に enable_categorical=True を明示的に追加
    dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
    dvalid = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
    dtest = xgb.DMatrix(X_te, enable_categorical=True)
    
    xgb_params = {'objective': 'binary:logistic', 'eval_metric': 'auc', 'learning_rate': 0.05, 'random_state': 42, 'tree_method': 'hist', 'max_depth': 6, 'enable_categorical': True}
    model_xgb = xgb.train(
        xgb_params, dtrain, num_boost_round=1000, evals=[(dtrain, 'train'), (dvalid, 'valid')],
        callbacks=[xgb.callback.EarlyStopping(rounds=50, save_best=True, maximize=True)], verbose_eval=False
    )
    models_info['XGBoost']['oof'][valid_idx] = model_xgb.predict(dvalid)
    models_info['XGBoost']['test'] += model_xgb.predict(dtest) / cv.n_splits
    joblib.dump(model_xgb, f"{save_dir}{prefix}xgboost_fold{fold+1}.pkl")

    # ----------------------------------------------------
    # 3/4: CatBoost
    # ----------------------------------------------------
    print("3/4: CatBoost...")
    X_tr_c, X_va_c, X_te_c = X_tr.copy(), X_va.copy(), X_te.copy()
    for col in cat_cols:
        X_tr_c[col] = X_tr_c[col].astype(str)
        X_va_c[col] = X_va_c[col].astype(str)
        X_te_c[col] = X_te_c[col].astype(str)
        
    model_cat = CatBoostClassifier(iterations=1000, learning_rate=0.05, eval_metric='AUC', random_seed=42, verbose=0, cat_features=cat_cols)
    model_cat.fit(X_tr_c, y_tr, eval_set=(X_va_c, y_va), early_stopping_rounds=50, verbose=False)
    
    models_info['CatBoost']['oof'][valid_idx] = model_cat.predict_proba(X_va_c)[:, 1]
    models_info['CatBoost']['test'] += model_cat.predict_proba(X_te_c)[:, 1] / cv.n_splits
    joblib.dump(model_cat, f"{save_dir}{prefix}catboost_fold{fold+1}.pkl")

    # ----------------------------------------------------
    # 4/4: Random Forest
    # ----------------------------------------------------
    print("4/4: Random Forest...")
    X_tr_r = X_train.iloc[train_idx]
    X_va_r = X_train.iloc[valid_idx]
    
    model_rf = RandomForestClassifier(n_estimators=500, max_depth=10, random_state=42, n_jobs=-1)
    model_rf.fit(X_tr_r, y_tr)
    
    models_info['RandomForest']['oof'][valid_idx] = model_rf.predict_proba(X_va_r)[:, 1]
    models_info['RandomForest']['test'] += model_rf.predict_proba(X_test)[:, 1] / cv.n_splits
    joblib.dump(model_rf, f"{save_dir}{prefix}randomforest_fold{fold+1}.pkl")

print("\n" + "="*40)
print(" すべてのモデルの学習・保存が完了しました！")
print("="*40)

# --- 5. 各モデル単体のCVスコア評価 ---
print("\n【検証データ（OOF）に対する ROC AUC スコア】")
for model_name, preds in models_info.items():
    score = roc_auc_score(y_train, preds['oof'])
    print(f"{model_name:<15} : {score:.5f}")
print("-" * 40)

# --- 6. アンサンブル（4モデルのブレンド） ---
oof_blend = (
    models_info['LightGBM']['oof'] * 0.35 +
    models_info['XGBoost']['oof'] * 0.30 +
    models_info['CatBoost']['oof'] * 0.25 +
    models_info['RandomForest']['oof'] * 0.10
)
score_blend = roc_auc_score(y_train, oof_blend)
print(f"★ 4モデルブレンドの 全体CV ROC-AUC: {score_blend:.5f}")
print("=" * 40)

# --- 7. 可視化・分析用のデータフレーム（df_errors）を作成 ---
df_errors = X_train.copy()
df_errors["Actual"] = y_train.values

for model_name, preds in models_info.items():
    df_errors[f"Pred_{model_name}"] = preds['oof']
    df_errors[f"Miss_{model_name}"] = ((df_errors[f"Pred_{model_name}"] >= 0.5).astype(int) != df_errors["Actual"]).astype(int)

miss_cols = [f"Miss_{m}" for m in models_info.keys()]
df_errors["Total_Miss_Count"] = df_errors[miss_cols].sum(axis=1)

os.makedirs("../../results/", exist_ok=True)
df_errors.to_csv("../../results/val_predictions_comparison.csv", index=False)
print("詳細データを '../../results/val_predictions_comparison.csv' に保存しました。")

# --- 8. 最終的な提出ファイルの作成 ---
final_preds = (
    models_info['LightGBM']['test'] * 0.35 +
    models_info['XGBoost']['test'] * 0.30 +
    models_info['CatBoost']['test'] * 0.25 +
    models_info['RandomForest']['test'] * 0.10
)

submission = pd.DataFrame({
    'id': test_ids,
    'Exited': final_preds
})
submission.to_csv('submission_ensemble_4models.csv', index=False)
print("submission_ensemble_4models.csv を保存しました！")

前処理を開始します...
前処理が完了しました。特徴量数: 17
=== 4大モデルの交差検証（5-Fold）を開始します ===

▼▼▼ Fold 1 / 5 の学習開始 ▼▼▼
1/4: LightGBM...
2/4: XGBoost...


c:\Users\ko.ameku.up\Desktop\kaggle_tutorial_zone\.venv_new\Lib\site-packages\xgboost\callback.py:385: UserWarning: [13:45:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()


3/4: CatBoost...
4/4: Random Forest...

▼▼▼ Fold 2 / 5 の学習開始 ▼▼▼
1/4: LightGBM...
2/4: XGBoost...


c:\Users\ko.ameku.up\Desktop\kaggle_tutorial_zone\.venv_new\Lib\site-packages\xgboost\callback.py:385: UserWarning: [13:45:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()


3/4: CatBoost...
4/4: Random Forest...

▼▼▼ Fold 3 / 5 の学習開始 ▼▼▼
1/4: LightGBM...
2/4: XGBoost...


...

3/4: CatBoost...
4/4: Random Forest...

▼▼▼ Fold 4 / 5 の学習開始 ▼▼▼
1/4: LightGBM...
2/4: XGBoost...


c:\Users\ko.ameku.up\Desktop\kaggle_tutorial_zone\.venv_new\Lib\site-packages\xgboost\callback.py:385: UserWarning: [13:46:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()


3/4: CatBoost...
4/4: Random Forest...

▼▼▼ Fold 5 / 5 の学習開始 ▼▼▼
1/4: LightGBM...
2/4: XGBoost...


c:\Users\ko.ameku.up\Desktop\kaggle_tutorial_zone\.venv_new\Lib\site-packages\xgboost\callback.py:385: UserWarning: [13:46:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()


3/4: CatBoost...
4/4: Random Forest...

 すべてのモデルの学習・保存が完了しました！

【検証データ（OOF）に対する ROC AUC スコア】
LightGBM        : 0.93392
XGBoost         : 0.93346
CatBoost        : 0.93597
RandomForest    : 0.93227
----------------------------------------
★ 4モデルブレンドの 全体CV ROC-AUC: 0.93529
詳細データを '../../results/val_predictions_comparison.csv' に保存しました。
submission_ensemble_4models.csv を保存しました！


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def preprocess_data(train_df, test_df):
    """
    銀行顧客離脱データセットの前処理および特徴量エンジニアリングを行う関数
    """
    print("前処理を開始します...")
    
    # 1. 正解ラベルの分離とIDの退避
    y_train = train_df['Exited'] if 'Exited' in train_df.columns else None
    
    # 学習用とテスト用を区別するためのフラグを立てて一時的に結合
    train_features = train_df.drop(columns=['Exited']) if 'Exited' in train_df.columns else train_df.copy()
    test_features = test_df.copy()
    
    train_features['is_train'] = 1
    test_features['is_train'] = 0
    
    combined = pd.concat([train_features, test_features], axis=0).reset_index(drop=True)
    
    # 2. 名字（Surname）の数（Frequency Encoding）
    # 同じ苗字がデータ中に何回登場するか（家族ルールの抽出）
    surname_counts = combined['Surname'].value_counts()
    combined['Surname_Freq'] = combined['Surname'].map(surname_counts)
    
    # 3. 製品数（NumOfProducts）の非線形性を捉えるフラグ
    # クロス集計で解約率が極端だった部分を明示的にフラグ化
    combined['Is_Product_2'] = (combined['NumOfProducts'] == 2).astype(int)
    combined['Is_High_Product'] = (combined['NumOfProducts'] >= 3).astype(int)
    
    # 4. 残高（Balance）に関する特徴量
    # 残高が0かどうかのフラグ
    combined['Is_Balance_Zero'] = (combined['Balance'] == 0).astype(int)
    # 年収に対する残高の割合（分母の0割りを防ぐため +1）
    combined['Balance_to_Salary_Ratio'] = combined['Balance'] / (combined['EstimatedSalary'] + 1)
    
    # 5. 年齢と信用スコアの掛け合わせ
    combined['CreditScore_by_Age'] = combined['CreditScore'] / combined['Age']
    

    
    # 7. カテゴリ変数のエンコーディング（Label Encoding）
    # GBDTで扱えるように文字列を数値（0, 1, 2...）に変換
    cat_cols = ['Geography', 'Gender']
    for col in cat_cols:
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col].astype(str))
    
    # 8. モデルの学習に不要なカラムの削除
    # 変換元のSurname、一意な値であるCustomerId、行識別用のidは削除（idは後で提出に使うため別途保持）
    drop_cols = ['id', 'CustomerId', 'Surname']
    combined = combined.drop(columns=[col for col in drop_cols if col in combined.columns])
    
    # 9. 再び train と test に分離
    X_train = combined[combined['is_train'] == 1].drop(columns=['is_train']).reset_index(drop=True)
    X_test = combined[combined['is_train'] == 0].drop(columns=['is_train']).reset_index(drop=True)
    
    print(f"前処理が完了しました。特徴量数: {X_train.shape[1]}")
    return X_train, y_train, X_test


In [2]:
# --- ここが原因！関数を実際に「実行」して変数を受け取るコードが必要です ---

# 1. まず元のデータを読み込む（パスはご自身の環境に合わせてください）
train = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\train.csv")
test = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\test.csv")
test_ids = test['id']

# 2. 定義した関数をここで呼び出す（これで X_train たちが誕生します）
X_train, y_train, X_test = preprocess_data(train, test)

前処理を開始します...
前処理が完了しました。特徴量数: 16


In [4]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# 必要モデルのインポート
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

# --- 1. データの読み込み ---
train = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\train.csv")
test = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\test.csv")
test_ids = test['id']

# --- 2. 前処理の実行 ---
X_train, y_train, X_test = preprocess_data(train, test)

# --- 3. クロスバリデーションの設定（Stratified K-Fold） ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 各モデルのOOF予測値とテスト予測値の格納用
models_info = {
    'LightGBM':     {'oof': np.zeros(len(X_train)), 'test': np.zeros(len(X_test))},
    'XGBoost':      {'oof': np.zeros(len(X_train)), 'test': np.zeros(len(X_test))},
    'CatBoost':     {'oof': np.zeros(len(X_train)), 'test': np.zeros(len(X_test))},
    'RandomForest': {'oof': np.zeros(len(X_train)), 'test': np.zeros(len(X_test))}
}

# カテゴリ変数のカラム名
cat_cols = ['Geography', 'Gender']

# 保存先の設定
save_dir = "../../model/"
os.makedirs(save_dir, exist_ok=True)
prefix = "05251045_"

print("=== 4大モデルの交差検証（5-Fold）を開始します ===")

# --- 4. 学習と予測のループ ---
for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n▼▼▼ Fold {fold + 1} / 5 の学習開始 ▼▼▼")
    
    # データの分割
    X_tr = X_train.iloc[train_idx].copy()
    y_tr = y_train.iloc[train_idx]
    X_va = X_train.iloc[valid_idx].copy()
    y_va = y_train.iloc[valid_idx]
    X_te = X_test.copy()
    
    # ----------------------------------------------------
    # 1/4: LightGBM
    # ----------------------------------------------------
    print("1/4: LightGBM...")
    for col in cat_cols:
        X_tr[col] = X_tr[col].astype('category')
        X_va[col] = X_va[col].astype('category')
        X_te[col] = X_te[col].astype('category')
        
    trn_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_va, label=y_va)
    
    model_lgb = lgb.train(
        {'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt', 'learning_rate': 0.05, 'random_state': 42, 'verbose': -1},
        trn_data, num_boost_round=1000, valid_sets=[trn_data, val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    models_info['LightGBM']['oof'][valid_idx] = model_lgb.predict(X_va, num_iteration=model_lgb.best_iteration)
    models_info['LightGBM']['test'] += model_lgb.predict(X_te, num_iteration=model_lgb.best_iteration) / cv.n_splits
    joblib.dump(model_lgb, f"{save_dir}{prefix}lightgbm_fold{fold+1}.pkl")

    # ----------------------------------------------------
    # 2/4: XGBoost
    # ----------------------------------------------------
    print("2/4: XGBoost...")
    # 【修正】DMatrixの引数に enable_categorical=True を明示的に追加
    dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
    dvalid = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
    dtest = xgb.DMatrix(X_te, enable_categorical=True)
    
    xgb_params = {'objective': 'binary:logistic', 'eval_metric': 'auc', 'learning_rate': 0.05, 'random_state': 42, 'tree_method': 'hist', 'max_depth': 6, 'enable_categorical': True}
    model_xgb = xgb.train(
        xgb_params, dtrain, num_boost_round=1000, evals=[(dtrain, 'train'), (dvalid, 'valid')],
        callbacks=[xgb.callback.EarlyStopping(rounds=50, save_best=True, maximize=True)], verbose_eval=False
    )
    models_info['XGBoost']['oof'][valid_idx] = model_xgb.predict(dvalid)
    models_info['XGBoost']['test'] += model_xgb.predict(dtest) / cv.n_splits
    joblib.dump(model_xgb, f"{save_dir}{prefix}xgboost_fold{fold+1}.pkl")

    # ----------------------------------------------------
    # 3/4: CatBoost
    # ----------------------------------------------------
    print("3/4: CatBoost...")
    X_tr_c, X_va_c, X_te_c = X_tr.copy(), X_va.copy(), X_te.copy()
    for col in cat_cols:
        X_tr_c[col] = X_tr_c[col].astype(str)
        X_va_c[col] = X_va_c[col].astype(str)
        X_te_c[col] = X_te_c[col].astype(str)
        
    model_cat = CatBoostClassifier(iterations=1000, learning_rate=0.05, eval_metric='AUC', random_seed=42, verbose=0, cat_features=cat_cols)
    model_cat.fit(X_tr_c, y_tr, eval_set=(X_va_c, y_va), early_stopping_rounds=50, verbose=False)
    
    models_info['CatBoost']['oof'][valid_idx] = model_cat.predict_proba(X_va_c)[:, 1]
    models_info['CatBoost']['test'] += model_cat.predict_proba(X_te_c)[:, 1] / cv.n_splits
    joblib.dump(model_cat, f"{save_dir}{prefix}catboost_fold{fold+1}.pkl")

    # ----------------------------------------------------
    # 4/4: Random Forest
    # ----------------------------------------------------
    print("4/4: Random Forest...")
    X_tr_r = X_train.iloc[train_idx]
    X_va_r = X_train.iloc[valid_idx]
    
    model_rf = RandomForestClassifier(n_estimators=500, max_depth=10, random_state=42, n_jobs=-1)
    model_rf.fit(X_tr_r, y_tr)
    
    models_info['RandomForest']['oof'][valid_idx] = model_rf.predict_proba(X_va_r)[:, 1]
    models_info['RandomForest']['test'] += model_rf.predict_proba(X_test)[:, 1] / cv.n_splits
    joblib.dump(model_rf, f"{save_dir}{prefix}randomforest_fold{fold+1}.pkl")

print("\n" + "="*40)
print(" すべてのモデルの学習・保存が完了しました！")
print("="*40)

# --- 5. 各モデル単体のCVスコア評価 ---
print("\n【検証データ（OOF）に対する ROC AUC スコア】")
for model_name, preds in models_info.items():
    score = roc_auc_score(y_train, preds['oof'])
    print(f"{model_name:<15} : {score:.5f}")
print("-" * 40)

# --- 6. アンサンブル（4モデルのブレンド） ---
oof_blend = (
    models_info['LightGBM']['oof'] * 0.35 +
    models_info['XGBoost']['oof'] * 0.30 +
    models_info['CatBoost']['oof'] * 0.25 +
    models_info['RandomForest']['oof'] * 0.10
)
score_blend = roc_auc_score(y_train, oof_blend)
print(f"★ 4モデルブレンドの 全体CV ROC-AUC: {score_blend:.5f}")
print("=" * 40)

# --- 7. 可視化・分析用のデータフレーム（df_errors）を作成 ---
df_errors = X_train.copy()
df_errors["Actual"] = y_train.values

for model_name, preds in models_info.items():
    df_errors[f"Pred_{model_name}"] = preds['oof']
    df_errors[f"Miss_{model_name}"] = ((df_errors[f"Pred_{model_name}"] >= 0.5).astype(int) != df_errors["Actual"]).astype(int)

miss_cols = [f"Miss_{m}" for m in models_info.keys()]
df_errors["Total_Miss_Count"] = df_errors[miss_cols].sum(axis=1)

os.makedirs("../../results/", exist_ok=True)
df_errors.to_csv("../../results/val_predictions_comparison.csv", index=False)
print("詳細データを '../../results/val_predictions_comparison.csv' に保存しました。")

# --- 8. 最終的な提出ファイルの作成 ---
final_preds = (
    models_info['LightGBM']['test'] * 0.35 +
    models_info['XGBoost']['test'] * 0.30 +
    models_info['CatBoost']['test'] * 0.25 +
    models_info['RandomForest']['test'] * 0.10
)

submission = pd.DataFrame({
    'id': test_ids,
    'Exited': final_preds
})
submission.to_csv('submission_ensemble_4models.csv', index=False)
print("submission_ensemble_4models.csv を保存しました！")

前処理を開始します...
前処理が完了しました。特徴量数: 16
=== 4大モデルの交差検証（5-Fold）を開始します ===

▼▼▼ Fold 1 / 5 の学習開始 ▼▼▼
1/4: LightGBM...
2/4: XGBoost...


c:\Users\ko.ameku.up\Desktop\kaggle_tutorial_zone\.venv_new\Lib\site-packages\xgboost\callback.py:385: UserWarning: [11:16:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()


3/4: CatBoost...
4/4: Random Forest...

▼▼▼ Fold 2 / 5 の学習開始 ▼▼▼
1/4: LightGBM...
2/4: XGBoost...


c:\Users\ko.ameku.up\Desktop\kaggle_tutorial_zone\.venv_new\Lib\site-packages\xgboost\callback.py:385: UserWarning: [11:16:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()


3/4: CatBoost...
4/4: Random Forest...

▼▼▼ Fold 3 / 5 の学習開始 ▼▼▼
1/4: LightGBM...
2/4: XGBoost...


...

3/4: CatBoost...
4/4: Random Forest...

▼▼▼ Fold 4 / 5 の学習開始 ▼▼▼
1/4: LightGBM...
2/4: XGBoost...


c:\Users\ko.ameku.up\Desktop\kaggle_tutorial_zone\.venv_new\Lib\site-packages\xgboost\callback.py:385: UserWarning: [11:17:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()


3/4: CatBoost...
4/4: Random Forest...

▼▼▼ Fold 5 / 5 の学習開始 ▼▼▼
1/4: LightGBM...
2/4: XGBoost...


c:\Users\ko.ameku.up\Desktop\kaggle_tutorial_zone\.venv_new\Lib\site-packages\xgboost\callback.py:385: UserWarning: [11:17:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()


3/4: CatBoost...
4/4: Random Forest...

 すべてのモデルの学習・保存が完了しました！

【検証データ（OOF）に対する ROC AUC スコア】
LightGBM        : 0.93376
XGBoost         : 0.93320
CatBoost        : 0.93599
RandomForest    : 0.93101
----------------------------------------
★ 4モデルブレンドの 全体CV ROC-AUC: 0.93511
詳細データを '../../results/val_predictions_comparison.csv' に保存しました。
submission_ensemble_4models.csv を保存しました！
